# ARGUS Text RPS Demo

Enter raw text, choose a retriever and thresholds, then run named-entity extraction plus ARGUS Retrieval Probability Score (RPS) scoring. The first run may download the NER or embedding model.

## Setup

In [23]:
from pathlib import Path
import importlib
import sys

repo_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "with_argus_eyes").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("Could not find the With_Argus_Eyes repository root from this notebook location.")
src_path = repo_root / "src"
for path in (repo_root, src_path):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = lambda value: value
    display = print

# Reload the local inference code so notebook kernels do not keep stale versions.
import with_argus_eyes.inference.text_risk as text_risk_module
import with_argus_eyes.inference as inference_module
importlib.reload(text_risk_module)
importlib.reload(inference_module)

from with_argus_eyes.inference import (
    ArgusTextConfig,
    analyze_text,
    available_retrievers,
    highlight_entities,
    resolve_model_artifact,
)

print("Using ARGUS inference module:", text_risk_module.__file__)
print("score_entities starts at line:", text_risk_module.score_entities.__code__.co_firstlineno)
print("Available retrievers:", ", ".join(available_retrievers()))


Using ARGUS inference module: /mounts/Users/cisintern/zeinabtaghavi/With_Argus_Eyes/src/with_argus_eyes/inference/text_risk.py
score_entities starts at line: 438
Available retrievers: contriever, qwen3, jina, bge-m3, reason-embed, nv-embed, gritlm, reasonir


## Environment configuration

In [24]:
import os

# Edit these before running the analysis cells.
# Use "" for CPU/default device behavior, or values such as "0", "0,1", or "6,7" for specific GPUs.
CUDA_VISIBLE_DEVICES = ""

# Keep Hugging Face downloads in a predictable location. Set to "" to use your system default.
HF_CACHE_DIR = str(repo_root / "outputs" / "cache" / "huggingface")

if CUDA_VISIBLE_DEVICES:
    os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
else:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)

if HF_CACHE_DIR:
    os.environ["HF_HOME"] = HF_CACHE_DIR
    os.environ["HF_HUB_CACHE"] = str(Path(HF_CACHE_DIR) / "hub")
    os.environ["HF_DATASETS_CACHE"] = str(Path(HF_CACHE_DIR) / "datasets")
    os.environ["TRANSFORMERS_CACHE"] = str(Path(HF_CACHE_DIR) / "transformers")

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<default/CPU>"))
print("HF_HOME:", os.environ.get("HF_HOME", "<system default>"))

CUDA_VISIBLE_DEVICES: <default/CPU>
HF_HOME: /mounts/Users/cisintern/zeinabtaghavi/With_Argus_Eyes/outputs/cache/huggingface


## User configuration

In [25]:
rps_threshold = 0.3

config = ArgusTextConfig(
    retriever="contriever",
    language="en",
    ner_model="dslim/bert-base-NER",
    risk_threshold=rps_threshold,  # Internal name kept for compatibility; this is the minimum acceptable RPS.
    ner_threshold=0.5,
    order=800,
    k=50,
    text_mode="span",
    workspace_root=repo_root,
)

artifact = resolve_model_artifact(config)
print("Selected ARGUS model artifact:")
print(artifact)

Selected ARGUS model artifact:
/mounts/Users/cisintern/zeinabtaghavi/With_Argus_Eyes/outputs/12_risk_outputs/contriever_ratio_unrelevant_below_k_50_o_800_k_50_sampled_average/models/mlp_best_contriever_S3_WD_Low_seed42.joblib


## Text input

In [26]:
text = """
St. Martin's Church in Zillis, Switzerland, is a Romanesque church best known
for its painted wooden ceiling panels dating from the 12th century. Neanderthals
inhabited Europe and Western and Central Asia during the Middle to Late Pleistocene.
""".strip()

print(text)

St. Martin's Church in Zillis, Switzerland, is a Romanesque church best known
for its painted wooden ceiling panels dating from the 12th century. Neanderthals
inhabited Europe and Western and Central Asia during the Middle to Late Pleistocene.


## Run analysis

In [27]:
results = analyze_text(text, config)

if not results:
    print("No named entities were found with the current NER threshold.")
else:
    print(f"Scored {len(results)} entity mentions.")

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


Scored 11 entity mentions.


## RPS results table

In [28]:
columns = ["entity", "entity_type", "ner_score", "rps_score", "meets_threshold", "below_threshold", "retriever"]

if results:
    try:
        import pandas as pd
        display(pd.DataFrame(results)[columns].sort_values("rps_score", ascending=False))
    except ImportError:
        for row in sorted(results, key=lambda item: item["rps_score"], reverse=True):
            print({key: row[key] for key in columns})
else:
    print("Nothing to display.")

,entity,entity_type,ner_score,rps_score,meets_threshold,below_threshold,retriever
7,Central Asia,LOC,0.997917,0.418508,True,False,contriever
8,Middle,MISC,0.993405,0.398147,True,False,contriever
6,Western,LOC,0.997865,0.365725,True,False,contriever
2,Switzerland,LOC,0.999736,0.363373,True,False,contriever
9,Late P,MISC,0.932711,0.282849,False,True,contriever
4,Neanderthal,MISC,0.868969,0.201108,False,True,contriever
10,##leistocene,MISC,0.833711,0.170215,False,True,contriever
3,Romanesque,MISC,0.990743,0.162075,False,True,contriever
0,St. Martin ' s Church,LOC,0.990153,0.139414,False,True,contriever
5,Europe,LOC,0.999537,0.130708,False,True,contriever


## Highlighted text

In [29]:
if results:
    display(HTML("<div style='line-height:1.8; font-size:1rem'>" + highlight_entities(text, results) + "</div>"))
else:
    print("No highlighted entities.")

## Compact JSON output

Entity names and Retrieval Probability Scores only.

In [30]:
import json

compact_results = [
    {"entity": row["entity"], "rps_score": float(row["rps_score"])}
    for row in sorted(results, key=lambda item: item["rps_score"], reverse=True)
]

print(json.dumps(compact_results, ensure_ascii=False, indent=2))


[
  {
    "entity": "Central Asia",
    "rps_score": 0.41850781440734863
  },
  {
    "entity": "Middle",
    "rps_score": 0.3981471061706543
  },
  {
    "entity": "Western",
    "rps_score": 0.3657246232032776
  },
  {
    "entity": "Switzerland",
    "rps_score": 0.36337345838546753
  },
  {
    "entity": "Late P",
    "rps_score": 0.2828490138053894
  },
  {
    "entity": "Neanderthal",
    "rps_score": 0.20110803842544556
  },
  {
    "entity": "##leistocene",
    "rps_score": 0.1702147126197815
  },
  {
    "entity": "Romanesque",
    "rps_score": 0.16207462549209595
  },
  {
    "entity": "St. Martin ' s Church",
    "rps_score": 0.13941441476345062
  },
  {
    "entity": "Europe",
    "rps_score": 0.13070805370807648
  },
  {
    "entity": "Zillis",
    "rps_score": 0.12152833491563797
  }
]
